In [ ]:
from netCDF4 import Dataset, MFDataset
import numpy as np
import numpy.ma as ma
import matplotlib.pyplot as plt
import xarray as xr 
import seaborn as sns
from matplotlib import rc
import matplotlib.cbook as cbook
import matplotlib.dates as dates
import matplotlib.ticker as ticker

In [ ]:
def calc_parc_o3(var):
    
        m_air = 28.964/(6.022E23)
        g = 980.6
    
        if 'plev' in var.dims: 
            plev=var.plev
            level='plev'
        else: 
            plev=var.lev
            level='lev'
        
        time=var.time
        delta_p = np.zeros((len(time), len(plev)))
        
        if plev[0]<plev[len(plev)-1] and plev[len(plev)-1] <= 1000: 
            factor=100
            factor_2=1 # conversion Pa->hPa
        if plev[0]>plev[len(plev)-1] and plev[0] <=1000: 
            factor=100
            factor_2=1
        if plev[0]<plev[len(plev)-1] and plev[len(plev)-1] >1000: 
            factor=1
            factor_2=100
        if plev[0]>plev[len(plev)-1] and plev[0] >1000: 
            factor=1
            factor_2=100
        
        if plev[0]<plev[len(plev)-1]: # for pressure levels in hPa
            
            for levelx in range(1,len(plev)): delta_p[:,levelx].fill(plev[levelx] - plev[levelx-1])

            weights_p = xr.DataArray(delta_p*factor, dims=['time',level], coords=[time,plev]) # difference between pressure levels in Pa
 
            O3 = var * weights_p * 10/ (g * m_air)
        
            if level=='plev': O3=O3.sel(plev=slice(30*factor_2,70*factor_2)) 
            else: O3=O3.sel(lev=slice(30*factor_2,70*factor_2))

            O3 = O3.sum(dim=level)
            O3 = O3/2.687E16  #calculate DU
            
        if plev[0]>plev[len(plev)-1]: # for pressure levels in hPa
            
            for levelx in range(0,len(plev)-1): delta_p[:,levelx].fill(plev[levelx] - plev[levelx+1])

            weights_p = xr.DataArray(delta_p*factor, dims=['time',level], coords=[time,plev]) # difference between pressure levels in Pa

            O3 = var * weights_p * 10/ (g * m_air)
            
            if level =='plev': O3=O3.sel(plev=slice(70*factor_2,30*factor_2)) 
            else: O3=O3.sel(lev=slice(70*factor_2,30*factor_2)) 
                
            O3 = O3.sum(dim=level)
            O3 = O3/2.687E16  #calculate DU
    
        return O3.where(O3 != 0)

In [ ]:
nc_fid_O3=xr.open_dataset('/net/hydro/chemie/mfriedel/Data/ozone_extremes/MERRA2/MERRA2_3d_dm_r144x96_1980-05.2020_zm.nc')

In [ ]:
var=nc_fid_O3['O3']
var=var*28.970/47.9982
var=var.mean(dim='lon')
var2=var
var=var.sel(time=var.time.dt.month.isin([12,1,2]))
var=var.mean(dim='time')

In [ ]:
var=var*var.lev
var=var.sel(lev=slice(1000,1))

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(8,5), constrained_layout=True, edgecolor='k',sharex=True, sharey=True)

levels=np.linspace(0,17.5, 22)

plot=ax.contourf(var.lat, np.log(var.lev), var*100*1000, levels=levels, cmap=plt.cm.get_cmap('RdBu_r'),extend='both')

ax.set_yticks([np.log(1000),np.log(100),np.log(10),np.log(1)])
ax.set_yticklabels(('1000','100','10','1'),fontsize=15)
ax.set_xticks([-80,-60,-40,-20,0,20,40,60,80])
ax.set_xticklabels(('-80','-60','-40','-20','0','20','40','60','80'), fontsize=15)
ax.invert_yaxis() 

cbar=fig.colorbar(plot, location='right')
cbar.ax.set_yticks([0,2.5,5.0,7.5,10.0,12.5,15.0,17.5])
cbar.ax.set_yticklabels(('0','2.5','5.0','7.5','10.0','12.5','15.0','17.5'),fontsize=15)
cbar.set_label('Ozone partial pressure (mPa)', fontsize=15)

ax.set_ylabel('Altitude (hPa)', fontsize=15)
ax.set_xlabel('Latitude (°N)', fontsize=15)

plt.savefig('ozone_partialpressure_MERRA2.pdf') 


In [ ]:

var=var2.sel(lat=slice(60,90))
weights = np.cos(np.deg2rad(var.lat))
var = var.weighted(weights)     
var=var.mean(dim='lat')

In [ ]:
var=calc_parc_o3(var)
var=var.sel(time=var.time.dt.month.isin([3,4]))
var=var.groupby('time.year').min()

In [ ]:
from matplotlib import rc

sns.set_style("ticks")
rc('font',**{'family':'sans-serif','sans-serif':['Titillium']})
rc('text', usetex=True)

fig, ax = plt.subplots(1,1,figsize=(7,5), constrained_layout=True, edgecolor='k',sharex='col')



var.plot(color='silver', linewidth=2.5)
ax.plot([2020,2011,2005,2002,2000,1997,1996,1995,1993,1990], [var.sel(year=2020),var.sel(year=2011),var.sel(year=2005),var.sel(year=2002),var.sel(year=2002),var.sel(year=1997),var.sel(year=1996),var.sel(year=1995),var.sel(year=1993),var.sel(year=1990)], marker='o', color='mediumvioletred', linestyle='', markersize=10)

sns.despine(offset=10, trim=True)
plt.ylabel('Partial ozone column (DU)', fontsize=20)
plt.xlabel('Year',fontsize=20)



ax.set_yticklabels(ax.get_yticks().astype(int), fontname='Titillium', fontsize=20)
ax.set_xticklabels(ax.get_xticks().astype(int), fontname='Titillium', fontsize=20)


plt.savefig('ozone_minima.pdf')  

In [ ]:
nc_fid_O3=xr.open_dataset('/net/hydro/chemie/mfriedel/Data/MERRA2/MERRA2_3D_UVT_r144x96_dm.nc')

In [ ]:
var=nc_fid_O3['T']

var=var.sel(lev=50)


varN=var.sel(lat=slice(60,90)).min(dim=['lat', 'lon'])

varS=var.sel(lat=slice(-90,-60)).min(dim=['lat','lon'])
             
             
#weights = np.cos(np.deg2rad(varN.lat))
#varN = varN.weighted(weights)     
#varN=varN.min(dim='lat')

In [ ]:

varN2018=varN.sel(time=var.time.dt.year.isin([2018])).groupby('time.dayofyear').mean()
varN2020=varN.sel(time=var.time.dt.year.isin([2020])).groupby('time.dayofyear').mean()


lower=varN.groupby('time.dayofyear').min()
upper=varN.groupby('time.dayofyear').max()
varN=varN.groupby('time.dayofyear').mean()


lowerS=varS.groupby('time.dayofyear').min()
upperS=varS.groupby('time.dayofyear').max()
varS=varS.groupby('time.dayofyear').mean()

In [ ]:
from matplotlib import rc
fig, ax = plt.subplots(1,1,figsize=(7,5), constrained_layout=True, edgecolor='k',sharex='col')

sns.set_style("ticks")
rc('font',**{'family':'sans-serif','sans-serif':['Titillium']})
rc('text', usetex=True)

varN.roll(dayofyear=184).plot(color='k')

#varN2018.roll(dayofyear=184).plot(color='mediumvioletred', label='2017 - 18')
varN2020.roll(dayofyear=184).plot(color='darkmagenta',label='2019 - 20')

plt.ylabel('Temperature (K)', fontsize=20)


ax.fill_between(varN.dayofyear, lower.roll(dayofyear=184), upper.roll(dayofyear=184), color='lightgrey')

ax.set_yticklabels(ax.get_yticks().astype(int), fontname='Titillium', fontsize=20)
ax.set_xticklabels(ax.get_xticks().astype(int), fontname='Titillium', fontsize=20)

ax.text(5,194, 'Polar stratospheric  \n clouds threshold'  , horizontalalignment='left',fontsize=20,color='royalblue')

plt.legend(loc='lower right', fontsize=20, frameon=False)
ax.set_ylim(170,230)
ax.set_xlim(0,366)


ax.xaxis.set_major_locator(dates.MonthLocator())
ax.xaxis.set_minor_locator(dates.MonthLocator(bymonthday=17))

ax.xaxis.set_major_formatter(ticker.NullFormatter())
ax.xaxis.set_minor_formatter(dates.DateFormatter('%b'))

for tick in ax.xaxis.get_minor_ticks():
    tick.tick1line.set_markersize(0)
    tick.tick2line.set_markersize(0)
    tick.label1.set_horizontalalignment('center')
    tick.label1.set_fontsize(18)

    
ax.set_xlabel('')


ax.tick_params('x', length=20, width=1, which='major')

plt.axhline(y=197, color='royalblue', linestyle='--')
ax.set_title("Arctic min. Temperature", fontsize=20)

plt.savefig('min_temp_MERRA2_2020.pdf')  

In [ ]:
var=nc_fid_O3['T']

var=var.sel(lev=50)


varN=var.sel(lat=slice(60,90)).min(dim=['lat', 'lon'])

varS=var.sel(lat=slice(-90,-60)).min(dim=['lat','lon'])
             
             
#weights = np.cos(np.deg2rad(varN.lat))
#varN = varN.weighted(weights)     
#varN=varN.min(dim='lat')

In [ ]:

varN2018=varN.sel(time=var.time.dt.year.isin([2018])).groupby('time.dayofyear').mean()
varN2020=varN.sel(time=var.time.dt.year.isin([2020])).groupby('time.dayofyear').mean()


lower=varN.groupby('time.dayofyear').min()
upper=varN.groupby('time.dayofyear').max()
varN=varN.groupby('time.dayofyear').mean()


lowerS=varS.groupby('time.dayofyear').min()
upperS=varS.groupby('time.dayofyear').max()
varS=varS.groupby('time.dayofyear').mean()

In [ ]:
from matplotlib import rc
import matplotlib.cbook as cbook
import matplotlib.dates as dates
import matplotlib.ticker as ticker

fig, ax = plt.subplots(1,2,figsize=(13,5), constrained_layout=True, edgecolor='k',sharex='col' )

sns.set_style("ticks")
rc('font',**{'family':'sans-serif','sans-serif':['Titillium']})
rc('text', usetex=True)

#varS.plot(color='k', ax=ax[0])

ax[0].set_ylabel('Temperature (K)', fontsize=20)

#ax[0].fill_between(varS.dayofyear, lowerS, upperS, color='lightgrey')

ax[0].set_yticklabels(ax[0].get_yticks().astype(int), fontname='Titillium', fontsize=20)
ax[0].set_xticklabels(ax[0].get_xticks().astype(int), fontname='Titillium', fontsize=20)

#ax[0].text(5,194, 'Cl activation \n threshold'  , horizontalalignment='left',fontsize=20,color='royalblue')

ax[0].legend(loc='lower right', fontsize=20, frameon=False)

ax[0].set_ylim(170,230)
ax[0].set_xlim(0,366)

ax[0].xaxis.set_major_locator(dates.MonthLocator())
ax[0].xaxis.set_minor_locator(dates.MonthLocator(bymonthday=17))

ax[0].xaxis.set_major_formatter(ticker.NullFormatter())
ax[0].xaxis.set_minor_formatter(dates.DateFormatter('%b'))

for tick in ax[0].xaxis.get_minor_ticks():
    tick.tick1line.set_markersize(0)
    tick.tick2line.set_markersize(0)
    tick.label1.set_horizontalalignment('center')
    tick.label1.set_fontsize(18)

    
ax[0].set_xlabel('')

ax[0].tick_params('x', length=20, width=1, which='major')

#ax[0].axhline(y=197, color='royalblue', linestyle='--')





varN.roll(dayofyear=184).plot(color='k', ax=ax[1])


time=np.concatenate((np.linspace(184,366,183), np.linspace(1,183,183)), axis=0)

#ax[1].plot(time, varN.roll(dayofyear=184), color='k')


#ax[1].plot(time[1:366], varN2018.roll(dayofyear=184),color='mediumvioletred', label='2017 - 18')
#ax[1].plot(time, varN2020.roll(dayofyear=184),color='darkmagenta', label='2019 - 20')

#varN2018.roll(dayofyear=184).plot(color='mediumvioletred', label='2017 - 18', ax=ax[1])
#varN2020.roll(dayofyear=184).plot(color='darkmagenta',label='2019 - 20', ax=ax[1])

ax[1].set_ylabel('Temperature (K)', fontsize=20)




ax[1].fill_between(lower.roll(dayofyear=184).dayofyear, lower.roll(dayofyear=184), upper.roll(dayofyear=184), color='lightgrey')


ax[1].text(5,194, 'Cl activation \n threshold'  , horizontalalignment='left',fontsize=20,color='royalblue')

ax[1].legend(loc='lower right', fontsize=20, frameon=False)
ax[1].set_ylim(170,230)
ax[1].set_xlim(0,366)


import datetime

date1 = datetime.date(2002, 7, 1)
date2 = datetime.date(2003, 6, 30)



ax[1].xaxis.set_major_locator(dates.MonthLocator())
ax[1].xaxis.set_minor_locator(dates.MonthLocator(bymonthday=17))

ax[1].xaxis.set_major_formatter(ticker.NullFormatter())
#ax[1].xaxis.set_minor_formatter(['Jul','Aug','Sep','Oct','Nov','Dec','Jan','Feb','Mar','Apr', 'May', 'Jun'])

ax[1].set_xticklabels(['Jul','Aug','Sep','Oct','Nov','Dec','Jan','Feb','Mar','Apr', 'May', 'Jun'], minor=True)



for tick in ax[1].xaxis.get_minor_ticks():
    tick.tick1line.set_markersize(0)
    tick.tick2line.set_markersize(0)
    tick.label1.set_horizontalalignment('center')
    tick.label1.set_fontsize(18)

    
ax[1].set_xlabel('')

ax[1].tick_params('x', length=20, width=1, which='major')

ax[1].axhline(y=197, color='royalblue', linestyle='--')
ax[1].set_title("Arctic min. Temperature", fontsize=20)

#ax[0].set_title("Antarctic min. Temperature", fontsize=20)

ax[1].set_yticklabels(ax[1].get_yticks().astype(int), fontname='Titillium', fontsize=20)


ax[0].axis('off')

plt.savefig('min_temp_MERRA2.pdf')  

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(6.5,5), constrained_layout=True, edgecolor='k',sharex='col' )



varN.roll(dayofyear=184).plot(color='k', ax=ax)


time=np.concatenate((np.linspace(184,366,183), np.linspace(1,183,183)))

#ax[1].plot(time, varN.roll(dayofyear=184), color='k')


#ax[1].plot(time[1:366], varN2018.roll(dayofyear=184),color='mediumvioletred', label='2017 - 18')
#ax[1].plot(time, varN2020.roll(dayofyear=184),color='darkmagenta', label='2019 - 20')

#varN2018.roll(dayofyear=184).plot(color='mediumvioletred', label='2017 - 18', ax=ax)
#varN2020.roll(dayofyear=184).plot(color='darkmagenta',label='2019 - 20', ax=ax)

ax.set_ylabel('Temperature (K)', fontsize=20)




ax.fill_between(lower.roll(dayofyear=184).dayofyear, lower.roll(dayofyear=184), upper.roll(dayofyear=184), color='lightgrey')


ax.text(5,194, 'Cl activation \n threshold'  , horizontalalignment='left',fontsize=20,color='royalblue')

ax.legend(loc='lower right', fontsize=20, frameon=False)
ax.set_ylim(170,230)
ax.set_xlim(0,366)


import datetime

date1 = datetime.date(2002, 7, 1)
date2 = datetime.date(2003, 6, 30)



ax.xaxis.set_major_locator(dates.MonthLocator())
ax.xaxis.set_minor_locator(dates.MonthLocator(bymonthday=17))

ax.xaxis.set_major_formatter(ticker.NullFormatter())
#ax[1].xaxis.set_minor_formatter(['Jul','Aug','Sep','Oct','Nov','Dec','Jan','Feb','Mar','Apr', 'May', 'Jun'])

ax.set_xticklabels(['Jul','Aug','Sep','Oct','Nov','Dec','Jan','Feb','Mar','Apr', 'May', 'Jun'], minor=True)



for tick in ax.xaxis.get_minor_ticks():
    tick.tick1line.set_markersize(0)
    tick.tick2line.set_markersize(0)
    tick.label1.set_horizontalalignment('center')
    tick.label1.set_fontsize(18)

    
ax.set_xlabel('')

ax.tick_params('x', length=20, width=1, which='major')

ax.axhline(y=197, color='royalblue', linestyle='--')
ax.set_title("Arctic min. Temperature", fontsize=20)

#ax[0].set_title("Antarctic min. Temperature", fontsize=20)

ax.set_yticklabels(ax.get_yticks().astype(int), fontname='Titillium', fontsize=20)


plt.savefig('min_temp_MERRA2_2.pdf')  

In [ ]:
var=nc_fid_O3['U']

var=var.sel(lev=10)


varN=var.sel(lat=slice(55,75)).mean(dim=['lat', 'lon'])

             
#weights = np.cos(np.deg2rad(varN.lat))
#varN = varN.weighted(weights)     
#varN=varN.min(dim='lat')

varN.time

In [ ]:

varN2020=varN.sel(time=slice('2019-07-01T09:00:00.000000000','2020-06-30T09:00:00.000000000')).groupby('time.dayofyear').mean()


varN2016=varN.sel(time=slice('2015-07-01T09:00:00.000000000','2016-06-30T09:00:00.000000000')).groupby('time.dayofyear').mean()

lower=varN.groupby('time.dayofyear').min()
upper=varN.groupby('time.dayofyear').max()
varN=varN.groupby('time.dayofyear').mean()



In [ ]:
from matplotlib import rc
fig, ax = plt.subplots(1,1,figsize=(7,5), constrained_layout=True, edgecolor='k',sharex='col')

sns.set_style("ticks")
rc('font',**{'family':'sans-serif','sans-serif':['Titillium']})
rc('text', usetex=True)

varN.roll(dayofyear=184).plot(color='k', label='1980 - 2020 mean')

varN2016.roll(dayofyear=184).plot(color='purple', label='2015 - 16')

#varN2020.roll(dayofyear=184).plot(color='mediumvioletred',label='2019 - 20')

plt.plot(range(len(varN2020)), varN2020.roll(dayofyear=184), color='mediumvioletred',label='2019 - 20')

plt.ylabel('Zonal wind (m/s)', fontsize=20)


ax.fill_between(varN.dayofyear, lower.roll(dayofyear=184), upper.roll(dayofyear=184), color='lightgrey')

ax.set_yticklabels(ax.get_yticks().astype(int), fontname='Titillium', fontsize=20)
ax.set_xticklabels(ax.get_xticks().astype(int), fontname='Titillium', fontsize=20)

ax.set_ylim(-40,65)

#ax.text(5,194, 'Polar stratospheric  \n clouds threshold'  , horizontalalignment='left',fontsize=20,color='royalblue')

plt.legend(loc='lower left', fontsize=20, frameon=False)
ax.set_xlim(0,366)

ax.xaxis.set_major_locator(dates.MonthLocator())
ax.xaxis.set_minor_locator(dates.MonthLocator(bymonthday=17))

ax.xaxis.set_major_formatter(ticker.NullFormatter())
#ax[1].xaxis.set_minor_formatter(['Jul','Aug','Sep','Oct','Nov','Dec','Jan','Feb','Mar','Apr', 'May', 'Jun'])

ax.set_xticklabels(['Jul','Aug','Sep','Oct','Nov','Dec','Jan','Feb','Mar','Apr', 'May', 'Jun'], minor=True)



for tick in ax.xaxis.get_minor_ticks():
    tick.tick1line.set_markersize(0)
    tick.tick2line.set_markersize(0)
    tick.label1.set_horizontalalignment('center')
    tick.label1.set_fontsize(18)
ax.set_xlabel('')


ax.tick_params('x', length=20, width=1, which='major')

plt.axhline(y=0, color='black', linestyle='--')
ax.set_title("Polar vortex strength", fontsize=20)

plt.savefig('vortex_MERRA2_III.pdf')  